In [0]:
from config import BRONZE_SCHEMA, SILVER_SCHEMA
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
custos_bronze = spark.table(f'{BRONZE_SCHEMA}.custos')
# display(custos_bronze)

In [0]:
custos_sem_duplicados = custos_bronze.drop('ingestion_datetime').dropDuplicates(['id_custo'])
_, passou = dq_check_unique('custos_sem_duplicados', 'id_custo único', custos_sem_duplicados, ['id_custo'])
if not passou:
    raise ValueError('Existem registros diferentes para o mesmo id_custo')
_, passou = dq_check('custos_sem_duplicados', 'id_custo não nulo', custos_sem_duplicados, F.col('id_custo').isNotNull())
if not passou:
    raise ValueError('Existem custos sem id_custo')


In [0]:
# Conversão da coluna custo para decimal
custos_silver = (
    custos_sem_duplicados
    .withColumn('custo', F.regexp_replace(F.trim(F.col('custo')), ',', '.'))
    .withColumn('custo', F.regexp_replace(F.col('custo'), '[^0-9.]', ''))
    .withColumn('custo', F.col('custo').cast('decimal(10, 2)'))
)
# display(custos_silver)

In [0]:
custos_silver.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_SCHEMA}.custos')